# 02 · Recommendation Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Souhaila-DJAFFAL/longtail-recsys/blob/main/notebooks/02_models.ipynb)

Three models of increasing capacity. Each exposes the same interface,
`score(users, history) → (batch, n_items + 1)` scores, so that evaluation code (notebook 03) treats
them identically. Column 0 is the padding item and is always masked out.

| Model | Signal it uses | Reference |
|---|---|---|
| **Popularity** | How often each item was consumed (non-personalized) | baseline |
| **BPR-MF** | Collaborative filtering: who interacted with what | Rendle et al., UAI 2009 |
| **SASRec** | Order of the user's recent interactions (self-attention) | Kang & McAuley, ICDM 2018 |

> This notebook is also loaded by notebooks 04 and 05 with `%run`, so the definitions live in one place.

In [1]:
# --- Setup: works locally (Jupyter) and on Google Colab ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("02_models.ipynb"):
    !git clone -q https://github.com/Souhaila-DJAFFAL/longtail-recsys.git
    %cd longtail-recsys/notebooks
    !pip install -q tabulate

DATASET = os.environ.get("RECSYS_DATASET", "ml-1m")  # "ml-1m" (real) or "synthetic" (fast check)
QUICK = os.environ.get("RECSYS_QUICK", "0") == "1"     # few epochs, for smoke tests only
ARTIFACTS, RESULTS = "../artifacts", "../results"
os.makedirs(ARTIFACTS, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)

In [2]:
import torch
import torch.nn as nn

## Popularity baseline

Ranks every item by its training popularity. It is a surprisingly strong accuracy baseline on
MovieLens, and the **most** popularity-biased recommender possible, which makes it a useful reference point.

In [3]:
class Popularity(nn.Module):
    def __init__(self, item_pop):
        super().__init__()
        self.register_buffer("pop", torch.as_tensor(item_pop, dtype=torch.float32))

    def score(self, users, hist):
        return self.pop.unsqueeze(0).expand(len(users), -1).clone()

## BPR-MF: matrix factorization with Bayesian Personalized Ranking

Each user $u$ and item $i$ get a $d$-dimensional embedding; the preference score is the dot product
$\hat{x}_{ui} = \mathbf{p}_u^\top \mathbf{q}_i$. Training uses triples $(u, i, j)$ where $i$ was consumed
by $u$ and $j$ is a sampled item that was not, and maximizes the probability that $u$ prefers $i$ over $j$:

$$\mathcal{L}_{\text{BPR}} = -\sum_{(u,i,j)} \log \sigma(\hat{x}_{ui} - \hat{x}_{uj}) + \lambda \lVert\Theta\rVert^2$$

It learns from co-occurrence patterns but ignores the order of interactions.

In [4]:
class BPRMF(nn.Module):
    def __init__(self, n_users, n_items, dim=64):
        super().__init__()
        self.user = nn.Embedding(n_users, dim)
        self.item = nn.Embedding(n_items + 1, dim, padding_idx=0)
        nn.init.normal_(self.user.weight, std=0.1)
        nn.init.normal_(self.item.weight, std=0.1)

    def forward(self, u, i, j):
        eu = self.user(u)
        return (eu * self.item(i)).sum(-1), (eu * self.item(j)).sum(-1)

    def score(self, users, hist):
        return self.user(users) @ self.item.weight.T

## SASRec: self-attentive sequential recommendation

SASRec reads the user's history as a sequence $(s_1, \dots, s_t)$ and predicts the next item with a
**causal Transformer encoder** (each position attends only to earlier positions):

1. Input: item embeddings + learned positional embeddings, dropout.
2. $L$ pre-norm self-attention blocks with a causal mask; padding positions are masked.
3. The hidden state at position $t$, $\mathbf{h}_t$, is scored against all item embeddings (shared with the
   input layer): $\hat{r}_{t,i} = \mathbf{h}_t^\top \mathbf{e}_i$.

Training predicts $s_{t+1}$ at every position with binary cross-entropy against one sampled negative per
position. At inference, the last hidden state ranks the whole catalogue. Because it models **order**, it
captures short-term intent (e.g., a user currently watching a series of thrillers).

In [5]:
class SASRec(nn.Module):
    def __init__(self, n_items, max_len=200, dim=64, n_blocks=2, n_heads=1, dropout=0.2):
        super().__init__()
        self.max_len, self.dim = max_len, dim
        self.item = nn.Embedding(n_items + 1, dim, padding_idx=0)
        self.pos = nn.Embedding(max_len, dim)
        self.drop = nn.Dropout(dropout)
        layer = nn.TransformerEncoderLayer(dim, n_heads, dim_feedforward=dim, dropout=dropout,
                                           activation="relu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_blocks, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(dim)
        for emb in (self.item, self.pos):
            nn.init.xavier_normal_(emb.weight)
        with torch.no_grad():
            self.item.weight[0].zero_()

    def encode(self, seq):
        """seq: (B, L) left-padded item ids -> hidden states (B, L, dim)."""
        L = seq.size(1)
        pos = torch.arange(L, device=seq.device).unsqueeze(0)
        pad = seq == 0
        x = self.drop(self.item(seq) * self.dim ** 0.5 + self.pos(pos)) * (~pad).unsqueeze(-1)
        causal = torch.triu(torch.ones(L, L, dtype=torch.bool, device=seq.device), diagonal=1)
        h = self.encoder(x, mask=causal, src_key_padding_mask=pad)
        h = torch.nan_to_num(h) * (~pad).unsqueeze(-1)   # fully padded rows -> 0
        return self.norm(h)

    def forward(self, seq, pos_items, neg_items):
        h = self.encode(seq)
        return (h * self.item(pos_items)).sum(-1), (h * self.item(neg_items)).sum(-1)

    def score(self, users, hist):
        h = self.encode(hist[:, -self.max_len:])[:, -1]   # most recent position
        return h @ self.item.weight.T

## Shape checks

In [6]:
_n_users, _n_items, _B, _L = 7, 30, 4, 12
_hist = torch.randint(1, _n_items + 1, (_B, _L)); _hist[:, :5] = 0          # left padding
_users = torch.arange(_B)
for _m in (Popularity(torch.arange(_n_items + 1)), BPRMF(_n_users, _n_items, 16),
           SASRec(_n_items, max_len=_L, dim=16)):
    _s = _m.score(_users, _hist)
    assert _s.shape == (_B, _n_items + 1) and torch.isfinite(_s).all()
    print(f"{type(_m).__name__:10s} score shape {tuple(_s.shape)}  "
          f"params {sum(p.numel() for p in _m.parameters()):,}")

Popularity score shape (4, 31)  params 0
BPRMF      score shape (4, 31)  params 608


SASRec     score shape (4, 31)  params 4,112


**Next:** [`03_evaluation_metrics.ipynb`](03_evaluation_metrics.ipynb) defines how the models are measured.